<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_01_04_hpc_bigdata_duckdb_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Big Data Processing with DuckDB in R


[DuckDB](https://duckdb.org/) is an open-source, high-performance, in-process analytical database management system designed for data analytics. It’s optimized for columnar data storage and query execution, making it ideal for analytical workloads like those in data science, machine learning, and business intelligence. Unlike traditional databases that run as separate server processes, DuckDB operates within the same process as the application, eliminating the need for a dedicated server and enabling fast, lightweight data processing. It supports SQL queries and integrates seamlessly with various programming languages, including R, Python, and Julia.

Key features of DuckDB:

* `In-process`: Runs embedded within the host application, reducing overhead.
* `Columnar storage`: Optimized for analytical queries with efficient columnar data processing.
* `SQL support`: Uses standard SQL for querying, with extensions for advanced analytics.
* `High performance`: Leverages modern hardware (e.g., multi-core CPUs, vectorized query execution).
* `Lightweight`: No external dependencies, easy to install and use.
* `Cross-platform`: Works on Windows, macOS, Linux, and more.
* `Integration`: Supports reading/writing from various formats like CSV, Parquet, and JSON.

DuckDB is particularly useful for scenarios where you need to analyze large datasets quickly without setting up a full-fledged database server, such as exploratory data analysis or prototyping.

## How DuckDB Works in R

DuckDB integrates with R through the duckdb R package, which provides an interface to connect to a `DuckDB` database, execute SQL queries, and work with data in R. This tutorial introduces the `DuckBD` package in R for handling large datasets that exceed RAM, using the NYC taxi dataset (yellow_tripdata_2023.csv) as an example. disk.frame stores data on disk in chunks, allowing efficient processing with familiar dplyr or data.table syntax.



### Installtion

To use `DuckDB` in R, you need to install the `{duckdb}` package. You can install it from CRAN or GitHub (for the latest development version).

```r
# Install from CRAN
install.packages("duckdb")

# Or install the development version from GitHub
# devtools::install_github("duckdb/duckdb-r")
```

Ensure you have a compatible R version (typically R 4.0 or later). The package includes the DuckDB engine, so no separate installation of DuckDB is required.



### Connecting to DuckDB

DuckDB can operate in two modes in R:

- `In-memory`: Data is stored in memory (default).
- `Persistent`: Data is stored on disk for persistence across sessions.

To connect to a DuckDB database, use the duckdb() function to create a connection. You can specify a file path for a persistent database or leave it blank for an in-memory database

```r
library(duckdb)

# Create an in-memory database
con <- dbConnect(duckdb::duckdb())

# Create a persistent database (stored in a file)
con <- dbConnect(duckdb::duckdb(), dbdir = "my_duckdb.db")
```r



### Working with Data

DuckDB in R allows you to:

- Execute SQL queries directly.
- Register R data frames as virtual tables.
- Read/write data from various sources (e.g., `CSV`, `Parquet`)



### Executing SQL Queries

You can run SQL queries using `dbGetQuery()` or `dbExecute()` from the `DBI` package, which `duckdb` supports.

```r
# Example: Create a table and insert data
dbExecute(con, "CREATE TABLE example (id INTEGER, name VARCHAR);")
dbExecute(con, "INSERT INTO example VALUES (1, 'Alice'), (2, 'Bob');")

# Query the table
result <- dbGetQuery(con, "SELECT * FROM example")
print(result)
#   id  name
# 1  1 Alice
# 2  2   Bob
```



### Registering R Data Frames

You can register an R data frame as a virtual table in DuckDB using `duckdb_register()`. This allows you to query the data frame using SQL without copying it into the database.

```r
# Create an R data frame
df <- data.frame(id = 1:3, name = c("Alice", "Bob", "Charlie"))

# Register the data frame as a table
duckdb_register(con, "mytable", df)

# Query the registered table
result <- dbGetQuery(con, "SELECT * FROM mytable WHERE id > 1")
print(result)
#   id  name
# 1  2   Bob
# 2  3 Charlie
```



### Reading/Writing External Files

DuckDB supports direct reading and writing of files like CSV and Parquet. You can query these files as if they were tables.


```r
# Write a data frame to a CSV file
write.csv(df, "data.csv", row.names = FALSE)

# Query the CSV file directly
result <- dbGetQuery(con, "SELECT * FROM 'data.csv' WHERE id > 1")
print(result)

# Export query results to a Parquet file
dbExecute(con, "COPY (SELECT * FROM mytable) TO 'output.parquet' (FORMAT PARQUET);")
```



### Key Features in R

`Integration with dplyr`: DuckDB works with the `dplyr` package, allowing you to use familiar `dplyr` syntax for data manipulation. The dbplyr package translates dplyr operations into SQL queries for DuckDB.

`Integration with duckplyr`: The `duckplyr` package is a drop-in replacement for `dplyr` that uses DuckDB as its backend to execute data manipulation operations

`Performance`: DuckDB’s vectorized query engine ensures fast execution, even for large datasets. It leverages R’s memory efficiently and supports parallel query execution.

`Memory Management`: For in-memory databases, data is stored in RAM. For large datasets, use a persistent database or query external files directly to manage memory usage.


### Closing the Connection

```r
dbDisconnect
----

### Advantages in R

- `Speed`: DuckDB is faster than many R-native data processing methods for large datasets due to its columnar storage and query optimization.

- `Flexibility`: Combines SQL’s power with R’s data manipulation ecosystem (dplyr, tidyr, etc.).

- `Scalability`: Handles large datasets by querying files directly or using persistent storage.

- `Ease of Use`: No need to set up a database server, making it ideal for quick analyses.



### Limitations

`In-memory Siz`e: For in-memory databases, data size is limited by available RAM. Use persistent databases or file-based queries for larger datasets.

`Learning Curve`: Requires familiarity with SQL for advanced queries, though dplyr integration mitigates this.

`Analytics Focus`: DuckDB is optimized for analytical (OLAP) workloads, not transactional (OLTP) tasks like frequent inserts/updates.

## Setup R in Python Runtime — Install {rpy2}

{rpy2} allows running R code in Colab’s Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there. Adjust `dataFolder` in the data-setup cell to point at your Parquet and CSV files (for example under `MyDrive`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')




## Data Processing with DuckDB in R

This tutorial introduces the `DuckBD` package in R for handling large datasets that exceed RAM, using the NYC taxi dataset (yellow_tripdata_2023.csv) as an example. disk.frame stores data on disk in chunks, allowing efficient processing with familiar dplyr or data.table syntax. We'll start from the basics (simple queries) and progress to advanced topics (complex SQL, joins, aggregations, and performance optimization).

### Check and Install Required R Packages

The cells below mirror the Quarto tutorial: define packages, install to Drive, verify, load.


### Define required packages


In [ ]:
%%R
packages <- c(
  "tidyverse",
  "data.table",
  "arrow",
  "duckdb",
  "duckplyr",
  "future",
  "ggplot2",
  "scales",
  "dbplyr",
  "conflicted",
  "lubridate"
)


### Install missing packages (Google Drive library)


In [ ]:
%%R
new.packages <- packages[!(packages %in% installed.packages(lib = "drive/My Drive/R/")[, "Package"])]
if (length(new.packages)) install.packages(new.packages, lib = "drive/My Drive/R/")


### Verify installation


In [ ]:
%%R
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load packages


In [ ]:
%%R
.libPaths(c("drive/My Drive/R/", .libPaths()))
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- packages


In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


```r
#| warning: false
#| error: false

# Install missing packages
new_packages <- packages[!(packages %in% installed.packages()[,"Package"])]
if(length(new_packages)) install.packages(new_packages)
```


##  Example Workflow

In [ ]:
%%R

# Connect to an in-memory database
con <- dbConnect(duckdb::duckdb())

# Create and register a data frame
df <- data.frame(
  id = 1:5,
  value = c(10, 20, 30, 40, 50),
  category = c("A", "B", "A", "B", "A")
)
duckdb_register(con, "data", df)

# Run a SQL query
result <- dbGetQuery(con, "
  SELECT category, AVG(value) as avg_value
  FROM data
  GROUP BY category
")

# Use dplyr for the same query
tbl(con, "data") %>%
  group_by(category) %>%
  summarise(avg_value = mean(value)) %>%
  collect()

# Export to Parquet
dbExecute(con, "COPY (SELECT * FROM data) TO 'data.parquet' (FORMAT PARQUET);")

# Disconnect
dbDisconnect(con, shutdown = TRUE)


### Data

This tutorial introduces `DuckDB` in R, using the NYC Yellow Taxi Trip Data for 2023 as an example dataset. `DuckDB` is an efficient, in-process analytical database that's perfect for handling large datasets like this one (which contains millions of rows for taxi trips). We'll start from the basics (installation and simple queries) and progress to advanced topics (complex SQL, joins, aggregations, and performance optimization).

The dataset is in Parquet format, which DuckDB can query directly without loading into memory. For reproducibility:
- Download the January 2023 data from:
[https://d37ci07v2hxiua.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet](https://d37ci07v2hxiua.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet) (about 47 MB, ~3 million rows).
- We saved  it to a local folder, e.g., `/home/zia207/Dropbox/WebSites/R_Website/Quarto_Projects/R_Beginner/Data/yellow_tripdata_2023-01.parquet`.
- We'll also use the Taxi Zone Lookup CSV for joins: Download from [https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv](https://s3.amazonaws.com/nyc-tlc/misc/taxi+_zone_lookup.csv).

The dataset includes columns like `VendorID`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `passenger_count`, `trip_distance`, `PULocationID`, `DOLocationID`, `fare_amount`, `total_amount`, etc.


### Set `dataFolder` for Colab (edit path)


In [ ]:
%%R
# Colab: set folder containing yellow_tripdata_2023-01.parquet and taxi_zone_lookup.csv
dataFolder <- file.path("/content/drive/MyDrive", "YourFolder", "Data")
# Ensure trailing slash
dataFolder <- sub("/$", "", dataFolder)
dataFolder <- paste0(dataFolder, "/")
parquet_file <- paste0(dataFolder, "yellow_tripdata_2023-01.parquet")
zone_file <- paste0(dataFolder, "taxi_zone_lookup.csv")


In [ ]:
%%R
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
parquet_file <- paste0(dataFolder, "yellow_tripdata_2023-01.parquet")
# Create an Arrow Dataset
ds <- open_dataset(parquet_file)


## Level 1: Basics

### Connect to DuckDB

Create an in-memory connection (fast for analysis; use `dbdir` for persistent storage if needed).


In [ ]:
%%R
con <- dbConnect(duckdb::duckdb())


### Register the Arrow Dataset in DuckDB

To query the `arrow` dataset (`ds`) with DuckDB, register it as a table/view using `duckdb_register_arrow()`.


In [ ]:
%%R
# Register the arrow dataset as a DuckDB table
duckdb_register_arrow(con, "taxi_data", ds)


### Query the Parquet File Directly

DuckDB can treat the Parquet file as a table without loading it fully. Use SQL to preview data.


In [ ]:
%%R
preview <- dbGetQuery(con, "SELECT * FROM taxi_data LIMIT 10")
print(preview)


In [ ]:
%%R
# Get column names and types (schema)
schema <- dbGetQuery(con, "DESCRIBE taxi_data")
print(schema)


### Basic Filtering

Filter trips with more than 1 passenger.


In [ ]:
%%R
filtered <- dbGetQuery(con, "SELECT * FROM taxi_data WHERE passenger_count > 1 LIMIT 10")
print(filtered)


### Disconnect

Always clean up.


In [ ]:
%%R
dbDisconnect(con, shutdown = TRUE)


## Level 2: Intermediate - Registering Data, dplyr Integration, and Aggregations

### Reconnect and Register the Dataset

Register the Parquet file as a virtual table for easier querying.


In [ ]:
%%R
con <- dbConnect(duckdb::duckdb())
duckdb_register_arrow(con, "taxi_data", ds)
# Register as a table
preview <- dbGetQuery(con, "SELECT * FROM taxi_data LIMIT 10")
print(preview)


### Simple Aggregations with SQL

Calculate average trip distance and total trips.


In [ ]:
%%R
agg_sql <- dbGetQuery(con, "
  SELECT
    AVG(trip_distance) AS avg_distance,
    COUNT(*) AS total_trips
  FROM taxi_data
")
print(agg_sql)


### Group By with SQL

Group by payment type and compute sums.


In [ ]:
%%R
group_sql <- dbGetQuery(con, "
  SELECT
    payment_type,
    SUM(total_amount) AS total_revenue,
    AVG(tip_amount) AS avg_tip
  FROM taxi_data
  GROUP BY payment_type
  ORDER BY total_revenue DESC
")
print(group_sql)


### Handling Dates

Extract pickup hour and aggregate.


In [ ]:
%%R
# Install and load the ICU extension
dbExecute(con, "INSTALL icu;")
dbExecute(con, "LOAD icu;")

# Now run your query
result <- dbGetQuery(con, "
  SELECT
    DATE_PART('hour', tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trip_count
  FROM taxi_data
  GROUP BY pickup_hour
  ORDER BY pickup_hour
")

# Print result
print(result)


In [ ]:
%%R
date_agg <- dbGetQuery(con, "
  SELECT
    DATE_PART('hour', tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trip_count
  FROM taxi_data
  GROUP BY pickup_hour
  ORDER BY pickup_hour
")
print(date_agg)
# Peak hours: Evenings show higher counts


In [ ]:
%%R
dbDisconnect(con, shutdown = TRUE)


## Level 3: Advanced - Joins, Window Functions, Multiple Files, and Performance

For advanced topics, we'll join with the taxi zone lookup, handle multiple months (assume you download February 2023 Parquet too: `yellow_tripdata_2023-02.parquet`), use window functions, and optimize.

### Download and Register Additional Data (`taxi_zone_lookup.csv)`

Download taxi zone CSV and save as `taxi_zone_lookup.csv` in `dataFolder`.


In [ ]:
%%R
zone_file <- paste0(dataFolder, "taxi_zone_lookup.csv")
con <- dbConnect(duckdb::duckdb())

# Register Parquet and CSV as views
dbExecute(con, paste0("CREATE VIEW taxi_data AS SELECT * FROM '", parquet_file, "'"))
dbExecute(con, paste0("CREATE VIEW zones AS SELECT * FROM '", zone_file, "'"))


###  Joins - Enrich Data with Locations

Join to get borough names for pickup locations.


In [ ]:
%%R
join_query <- dbGetQuery(con, "
  SELECT
    t.VendorID,
    t.trip_distance,
    t.total_amount,
    z.Borough AS pickup_borough
  FROM taxi_data t
  LEFT JOIN zones z ON t.PULocationID = z.LocationID
  WHERE t.trip_distance > 10
  LIMIT 10
")
print(join_query)


Group by borough for aggregations.


In [ ]:
%%R
borough_agg <- dbGetQuery(con, "
  SELECT
    z.Borough AS pickup_borough,
    AVG(t.trip_distance) AS avg_distance,
    SUM(t.total_amount) AS total_revenue
  FROM taxi_data t
  LEFT JOIN zones z ON t.PULocationID = z.LocationID
  GROUP BY pickup_borough
  ORDER BY total_revenue DESC
")
print(borough_agg)
# Manhattan likely tops revenue


In [ ]:
%%R
# Create bar chart
ggplot(borough_agg, aes(x = reorder(pickup_borough, -total_revenue), y = total_revenue)) +
  geom_bar(stat = "identity", fill = "skyblue") +
  labs(
    title = "Total Revenue by Pickup Borough (January 2023)",
    x = "Borough",
    y = "Total Revenue ($)"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_text(hjust = 0.5)
  ) +
  scale_y_continuous(labels = scales::dollar_format())


### Window Functions

Rank trips by distance within each payment type.


In [ ]:
%%R
window_query <- dbGetQuery(con, "
  SELECT
    payment_type,
    trip_distance,
    RANK() OVER (PARTITION BY payment_type ORDER BY trip_distance DESC) AS distance_rank
  FROM taxi_data
  LIMIT 20
")
print(window_query)


### SQuerying Multiple Files (e.g., Jan + Feb 2023)


In [ ]:
%%R
dbExecute(con, "INSTALL httpfs; LOAD httpfs;")
multi_data <- dbGetQuery(con, "
  SELECT * FROM read_parquet([
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet',
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet'
  ])
") %>% as_tibble()
total_trips <- multi_data %>%
  summarise(total_trips = n())
print(total_trips)


###  Export Results

Write query results to CSV or Parquet.


In [ ]:
%%R
duckdb_register_arrow(con, "taxi_data", ds)
dbExecute(con, paste0("COPY (SELECT * FROM taxi_data WHERE trip_distance > 5 LIMIT 1000) TO '", dataFolder, "output.csv' (FORMAT CSV, HEADER)"))


In [ ]:
%%R
dbDisconnect(con, shutdown = TRUE)


## Integrate with dplyr

`DuckDB` works with the `dplyr` package, allowing you to use familiar `dplyr` syntax for data manipulation. The dbplyr package translates `dplyr` operations into SQL queries for DuckDB.

This tutorial uses `dplyr` functions with the NYC Yellow Taxi Trip Data (`yellow_tripdata_2023.parquet`, ~3 million rows), integrating `arrow` for efficient data loading and `duckdb` for SQL operations.

### Load and Register Arrow Dataset


In [ ]:
%%R
ds <- open_dataset(parquet_file)
con <- dbConnect(duckdb::duckdb())
duckdb_register_arrow(con, "taxi_data", ds)
taxi_df <- tbl(con, "taxi_data")  # dplyr table reference


In [ ]:
%%R
preview <- taxi_df %>%
  head(10) %>%
  collect()
print(preview)


### Basic Filtering


In [ ]:
%%R
filtered <- taxi_df %>%
  filter(passenger_count > 1) %>%
  head(10) %>%
  collect()
print(filtered)


### Aggregations and Grouping


In [ ]:
%%R
# Simple aggregation
agg_result <- taxi_df %>%
  summarise(
    avg_distance = mean(trip_distance, na.rm = TRUE),
    total_trips = n()
  ) %>%
  collect()
print(agg_result)


In [ ]:
%%R
# Grouping by Payment Type
group_result <- taxi_df %>%
  group_by(payment_type) %>%
  summarise(
    total_revenue = sum(total_amount, na.rm = TRUE),
    avg_tip = mean(tip_amount, na.rm = TRUE)
  ) %>%
  arrange(desc(total_revenue)) %>%
  collect()
print(group_result)


In [ ]:
%%R
# Create a tbl reference
taxi_tbl <- tbl(con, "taxi_data")

# dplyr pipeline: Filter, group, summarize
result_dplyr <- taxi_tbl %>%
  filter(trip_distance > 5) %>%
  group_by(payment_type) %>%
  summarise(
    avg_fare = mean(fare_amount, na.rm = TRUE),
    total_trips = n()
  ) %>%
  arrange(desc(total_trips)) %>%
  collect()  # Pull results into R memory

print(result_dplyr)


In [ ]:
%%R
# handling date
hourly_trips <- taxi_df %>%
  mutate(pickup_hour = hour(tpep_pickup_datetime)) %>%
  group_by(pickup_hour) %>%
  summarise(trip_count = n()) %>%
  arrange(pickup_hour) %>%
  collect()
print(head(hourly_trips))


### Joins


In [ ]:
%%R
# Load Tazixi Zone Lookup
zone_file <- paste0(dataFolder, "taxi_zone_lookup.csv")
zones_df <- read.csv(zone_file) %>% as_tibble()
dbExecute(con, paste0("CREATE VIEW zones AS SELECT * FROM '", zone_file, "'"))


In [ ]:
%%R
# Join
joined_result <- taxi_df %>%
  left_join(tbl(con, "zones"), by = c("PULocationID" = "LocationID")) %>%
  filter(trip_distance > 10) %>%
  select(VendorID, trip_distance, total_amount, Borough) %>%
  head(10) %>%
  collect()
print(joined_result)


In [ ]:
%%R
# Group by borough
borough_agg <- taxi_df %>%
  left_join(tbl(con, "zones"), by = c("PULocationID" = "LocationID")) %>%
  group_by(Borough) %>%
  summarise(
    avg_distance = mean(trip_distance, na.rm = TRUE),
    total_revenue = sum(total_amount, na.rm = TRUE)
  ) %>%
  arrange(desc(total_revenue)) %>%
  collect()
print(borough_agg)


### Windows Functions


In [ ]:
%%R
running_total <- taxi_df %>%
  mutate(pickup_date = as_date(tpep_pickup_datetime)) %>%
  group_by(pickup_date) %>%
  summarise(daily_revenue = sum(total_amount, na.rm = TRUE)) %>%
  arrange(pickup_date) %>%
  mutate(running_revenue = cumsum(daily_revenue)) %>%
  collect()
print(running_total)


In [ ]:
%%R
running_total <- taxi_df %>%
  mutate(pickup_date = as_date(tpep_pickup_datetime)) %>%
  group_by(pickup_date) %>%
  summarise(daily_revenue = sum(total_amount, na.rm = TRUE)) %>%
  arrange(pickup_date) %>%
  mutate(running_revenue = cumsum(daily_revenue)) %>%
  collect()
print(running_total)


### Export Results


In [ ]:
%%R
filtered_df <- taxi_df %>%
  filter(trip_distance > 5) %>%
  head(1000) %>%
  collect()
write.csv(filtered_df, paste0(dataFolder, "output.csv"), row.names = FALSE)


In [ ]:
%%R
### Disconnect
dbDisconnect(con, shutdown = TRUE)


## Integrate with duckplyr

The [`duckplyr`](https://github.com/tidyverse/duckplyr) package is a drop-in replacement for dplyr that uses DuckDB as its backend to execute data manipulation operations, offering significant performance improvements for large datasets by leveraging DuckDB's efficient columnar storage and query engine. Unlike dplyr, which processes data in R's memory, duckplyr translates dplyr-style code into SQL queries executed by DuckDB, often resulting in faster operations, especially for large datasets like the NYC Yellow Taxi Trip Data


Calling `library(duckplyr)` overwrites dplyr methods, enabling duckplyr for the entire session:


In [ ]:
%%R
library(conflicted)
library(duckplyr)

conflict_prefer("filter", "dplyr")


### Load Data as a duckplyr Data Frame

With `duckplyr`, you can load the Parquet file directly into a duckplyr data frame, which is a wrapper around a DuckDB table or view. Use `duckplyr::as_duckdb_tibble()` to create a duckplyr data frame from a Parquet file.


In [ ]:
%%R
# Set data path
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"
parquet_file <- paste0(dataFolder, "yellow_tripdata_2023.parquet")
zone_file <- paste0(dataFolder, "taxi_zone_lookup.csv")


In [ ]:
%%R
# Load and register Arrow dataset
ds <- open_dataset(parquet_file)
con <- dbConnect(duckdb::duckdb())
duckdb_register_arrow(con, "taxi_data", ds)
taxi_df <- tbl(con, "taxi_data")  # Use dbplyr table


In [ ]:
%%R
# Register zones as DuckDB table
zones_df <- read.csv(zone_file)
duckdb_register(con, "zones", zones_df)
zones_tbl <- tbl(con, "zones")


In [ ]:
%%R
# Register zones
zones_df <- as_duckplyr_df(read.csv(zone_file))
dbExecute(con, paste0("CREATE VIEW zones AS SELECT * FROM '", zone_file, "'"))


In [ ]:
%%R
# Verify table registration
dbGetQuery(con, "SHOW TABLES")


In [ ]:
%%R
# Preview data
preview <- taxi_df %>%
  head(10) %>%
  collect()
print(preview)


In [ ]:
%%R
# Inspect schema
glimpse(taxi_df)


### Basic Filtering


In [ ]:
%%R
# Basic filtering: Trips with more than 1 passenger
filtered <- taxi_df %>%
  filter(passenger_count > 1) %>%
  head(10) %>%
  collect()
print(filtered)


### Aggregations and Grouping


In [ ]:
%%R
# Simple aggregations
agg_result <- taxi_df %>%
  summarise(
    avg_distance = mean(trip_distance, na.rm = TRUE),
    total_trips = n()
  ) %>%
  collect()
print(agg_result)


In [ ]:
%%R
# Grouping by payment type
group_result <- taxi_df %>%
  group_by(payment_type) %>%
  summarise(
    total_revenue = sum(total_amount, na.rm = TRUE),
    avg_tip = mean(tip_amount, na.rm = TRUE)
  ) %>%
  arrange(desc(total_revenue)) %>%
  collect()
print(group_result)


In [ ]:
%%R
# Handling dates: Count trips by pickup hour
hourly_trips <- taxi_df %>%
  mutate(pickup_hour = lubridate::hour(tpep_pickup_datetime)) %>%
  group_by(pickup_hour) %>%
  summarise(trip_count = n()) %>%
  arrange(pickup_hour) %>%
  collect()
print(hourly_trips)


### Joins, Window Functions, Exporting, and Larger-than-Memory Data


In [ ]:
%%R
# Join: Long trips with borough information
joined_result <- taxi_df %>%
  left_join(zones_tbl, by = c("PULocationID" = "LocationID")) %>%
  filter(trip_distance > 10) %>%
  select(VendorID, trip_distance, total_amount, Borough) %>%
  head(10) %>%
  collect()
print(joined_result)


In [ ]:
%%R
# Group by borough (for bar chart)
borough_agg <- taxi_df %>%
  left_join(zones_tbl, by = c("PULocationID" = "LocationID")) %>%
  group_by(pickup_borough = Borough) %>%
  summarise(
    avg_distance = mean(trip_distance, na.rm = TRUE),
    total_revenue = sum(total_amount, na.rm = TRUE)
  ) %>%
  arrange(desc(total_revenue)) %>%
  collect()
print(borough_agg)


In [ ]:
%%R
# Bar chart of total revenue by borough
library(ggplot2)
ggplot(borough_agg, aes(x = reorder(pickup_borough, -total_revenue), y = total_revenue)) +
  geom_bar(stat = "identity", fill = "skyblue") +
  labs(
    title = "Total Revenue by Pickup Borough (January 2023)",
    x = "Borough",
    y = "Total Revenue ($)"
  ) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_text(hjust = 0.5)
  ) +
  scale_y_continuous(labels = scales::dollar_format())


In [ ]:
%%R
# Window function: Running total revenue by date
running_total <- taxi_df %>%
  mutate(pickup_date = as_date(tpep_pickup_datetime)) %>%
  group_by(pickup_date) %>%
  summarise(daily_revenue = sum(total_amount, na.rm = TRUE)) %>%
  arrange(pickup_date) %>%
  mutate(running_revenue = cumsum(daily_revenue)) %>%
  collect()
print(running_total)


### Export Results


In [ ]:
%%R
# Export results: Trips with trip_distance > 5
taxi_df %>%
  filter(trip_distance > 5) %>%
  head(1000) %>%
  dbplyr::sql_render() %>%
  {dbExecute(con, paste0("COPY (", ., ") TO '", dataFolder, "output.csv' (FORMAT CSV, HEADER)"))}


### Analyzing larger-than-memory data (remote Parquet files)


In [ ]:
%%R
# Analyzing larger-than-memory data (remote Parquet files)
dbExecute(con, "INSTALL httpfs; LOAD httpfs;")
remote_urls <- c(
  "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet",
  "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet"
)
remote_df <- as_duckplyr_df(dbGetQuery(con, paste0("SELECT * FROM read_parquet(['", paste(remote_urls, collapse = "', '"), "'])")))
total_trips <- remote_df %>%
  summarise(total_trips = n()) %>%
  collect()
print(total_trips)


In [ ]:
%%R
# Disconnect
dbDisconnect(con, shutdown = TRUE)


In [ ]:
%%R
# Disconnect
if (dbIsValid(con)) dbDisconnect(con, shutdown = TRUE)


## Basic Data Pre-processing with DuckDB

DuckDB is not a machine learning library, but it plays a powerful supporting role in ML workflows—especially in the data preprocessing, feature engineering, and scalable inference stages. By performing these steps inside the database, you avoid costly data movement, reduce memory pressure, and often achieve significant performance gains over traditional R-based pipelines.


 ### Why Use DuckDB for ML?

| Benefit | Explanation |
|--------|-------------|
| **Speed** | Vectorized, parallel SQL execution often outperforms Pandas/scikit-learn on large data |
| **Memory Efficiency** | Processes data in chunks; no need to load full dataset into RAM |
| **Reproducibility** | Preprocessing logic lives in SQL—easy to version, audit, and reuse |
| **Data Locality** | Transform data where it lives (Parquet, CSV, S3, etc.) |
| **Consistency** | Same SQL logic can be used at training **and** inference time |


Below is a **complete R translation** of the [Basic Feature Engineering with DuckDB](https://duckdb.org/2025/08/15/ml-data-preprocessing.html?spm=a2ty_o01.29997173.0.0.619ac9217IETtc) blog post, using **`duckdb`** and **`dplyr`-like syntax via `dbplyr`** (which integrates with `dplyr` for SQL generation).

We'll replicate the key steps:

- Load data
- Summarize
- One-hot, Ordinal, and Label Encoding
- Train/test split
- Feature scaling (Standard, Min-Max, Robust)
- Missing value imputation

### Load Data into DuckDB

We will be working with a synthetic financial transactions dataset from [Kaggle](https://www.kaggle.com/datasets/aryan208/financial-transactions-dataset-for-fraud-detection/data), which contains generic information used for fraud detection in financial transactions.


In [ ]:
%%R
# Connect to an in-memory DuckDB database
con <- dbConnect(duckdb())

# Install and load ICU extension (needed for date/time functions)
dbExecute(con, "INSTALL icu;")
dbExecute(con, "LOAD icu;")

# Read CSV directly from URL into a DuckDB table
dbExecute(con, "
  CREATE TABLE financial_trx AS
  FROM read_csv('https://blobs.duckdb.org/data/financial_fraud_detection_dataset.csv')
")


###  Summarize the Data (Equivalent to `SUMMARIZE`)

DuckDB's `SUMMARIZE` isn't available in R directly, but we can replicate it.


In [ ]:
%%R
summary_stats <- tbl(con, sql("
  SELECT
    column_name,
    column_type,
    count,
    null_percentage,
    min
  FROM (SUMMARIZE financial_trx)
"))

# Collect and print
summary_stats %>% collect()


>  This uses raw SQL inside `sql()` because `SUMMARIZE` is a DuckDB-specific meta-command.

### Feature Encoding

The summary statistics reveal several categorical columns in the dataset, including `transaction_type`, `merchant_category`, and `payment_channel`. Since most machine learning algorithms require numerical input, these categorical variables need to be transformed into numerical form. This transformation process is known as **encoding**, and there are various methods to achieve it. Below, we demonstrate some widely used encoding techniques using SQL.

### One-Hot Encoding

One-hot encoding creates binary columns for each category in a categorical variable. For example, the `transaction_type` column has four unique values: `deposit`, `payment`, `transfer`, and `withdrawal`. One-hot encoding will create four new binary columns, each indicating the presence (1) or absence (0) of a specific transaction type.


#### Option 1: Manual CASE-style one-hot encoding


In [ ]:
%%R
one_hot_manual <- tbl(con, "financial_trx") %>%
  select(transaction_type) %>%
  distinct() %>%
  mutate(
    deposit_onehot = sql("CAST(transaction_type = 'deposit' AS INT)"),
    payment_onehot = sql("CAST(transaction_type = 'payment' AS INT)"),
    transfer_onehot = sql("CAST(transaction_type = 'transfer' AS INT)"),
    withdrawal_onehot = sql("CAST(transaction_type = 'withdrawal' AS INT)")
  )

one_hot_manual %>% show_query()
one_hot_manual %>% collect()


#### Option 2: Use `PIVOT` via SQL (not directly supported in `dplyr`, so use raw SQL)


In [ ]:
%%R
one_hot_pivot <- tbl(con, sql("
  PIVOT financial_trx ON transaction_type
  USING COALESCE(MAX(transaction_type = transaction_type)::INT, 0) AS onehot
  GROUP BY transaction_type
"))

one_hot_pivot %>% collect()


### Ordinal Encoding

Ordinal encoding assigns a unique integer to each category based on a specified order. For `transaction_type`, we can assign integers as follows: `deposit` = 0, `payment` = 1, `transfer` = 2, `withdrawal` = 3. This encoding is useful when there is an inherent order in the categories.


In [ ]:
%%R
trx_type_ordinal_encoded <- tbl(con, sql("
  SELECT
    transaction_type,
    ROW_NUMBER() OVER (ORDER BY transaction_type) - 1 AS trx_type_oe
  FROM (SELECT DISTINCT transaction_type FROM financial_trx)
"))

# Join back to full data
financial_trx_with_oe <- tbl(con, "financial_trx") %>%
  inner_join(trx_type_ordinal_encoded, by = "transaction_type")

financial_trx_with_oe %>%
  group_by(transaction_type, trx_type_oe) %>%
  summarise(number_trx = n(), .groups = "drop") %>%
  arrange(trx_type_oe) %>%
  collect()


### Label Encoding (Non-deterministic order)

Similarly to ordinal encoding, label encoding assigns a unique identifier, but it does not take order into account and it is, usually, applied to output data:


In [ ]:
%%R
trx_type_label_encoded <- tbl(con, sql("
  SELECT
    transaction_type,
    ROW_NUMBER() OVER () - 1 AS trx_type_le
  FROM (SELECT DISTINCT transaction_type FROM financial_trx)
"))

financial_trx_with_le <- tbl(con, "financial_trx") %>%
  inner_join(trx_type_label_encoded, by = "transaction_type")

financial_trx_with_le %>%
  group_by(transaction_type, trx_type_le) %>%
  summarise(number_trx = n(), .groups = "drop") %>%
  arrange(trx_type_le) %>%
  collect()


### Train/Test Split (Reservoir Sampling)


In [ ]:
%%R
# Set single-threaded for reproducibility
dbExecute(con, "SET threads = 1;")

# Create training sample (80%)
dbExecute(con, "
  CREATE OR REPLACE TABLE financial_trx_training AS
  FROM financial_trx USING SAMPLE 80 PERCENT (reservoir, 256)
")

# Restore thread count
dbExecute(con, "SET threads = 8;")

# Create testing set (anti-join)
dbExecute(con, "
  CREATE OR REPLACE TABLE financial_trx_testing AS
  FROM financial_trx ANTI JOIN financial_trx_training USING (transaction_id)
")


### Feature Scaling

Another essential step in machine learning preprocessing is scaling numerical features to ensure they fall within a comparable range or distribution. Known as feature normalization or standardization, this process adjusts feature magnitudes so they are on a similar scale—either by rescaling values to a specific interval (such as 0 to 1) or by transforming them to have a mean of zero and a standard deviation of one. Scaling is crucial because many machine learning algorithms use distance-based calculations or gradient-driven optimization, both of which can be distorted when features have vastly different scales.


In [ ]:
%%R
# Create standard scaler macro
dbExecute(con, "
  CREATE OR REPLACE MACRO standard_scaler(val, avg_val, std_val) AS
    (val - avg_val) / std_val
")


In [ ]:
%%R
# Create min-max scaler
dbExecute(con, "
  CREATE OR REPLACE MACRO min_max_scaler(val, min_val, max_val) AS
    (val - min_val) / NULLIF(max_val - min_val, 0)
")


In [ ]:
%%R
# Create robust scaler
dbExecute(con, "
  CREATE OR REPLACE MACRO robust_scaler(val, q25_val, q50_val, q75_val) AS
    (val - q50_val) / NULLIF(q75_val - q25_val, 0)
")


> ⚠️ Note: The above macro uses dynamic SQL and array unnesting. For simplicity, we’ll compute scaling parameters per column below.



###  ompute Scaling Parameters (Practical Approach)

Instead of complex macros, here's a **practical R + DuckDB** way:


In [ ]:
%%R
# Compute scaling parameters from training set
scaling_params <- tbl(con, "financial_trx_training") %>%
  summarise(
    avg_velocity_score = mean(velocity_score),
    std_velocity_score = sd(velocity_score),
    min_velocity_score = min(velocity_score),
    max_velocity_score = max(velocity_score),
    q25_velocity_score = quantile(velocity_score, 0.25),
    q50_velocity_score = quantile(velocity_score, 0.50),
    q75_velocity_score = quantile(velocity_score, 0.75),
    median_velocity_score = median(velocity_score),

    avg_spending_deviation_score = mean(spending_deviation_score),
    std_spending_deviation_score = sd(spending_deviation_score),
    min_spending_deviation_score = min(spending_deviation_score),
    max_spending_deviation_score = max(spending_deviation_score),
    q25_spending_deviation_score = quantile(spending_deviation_score, 0.25),
    q50_spending_deviation_score = quantile(spending_deviation_score, 0.50),
    q75_spending_deviation_score = quantile(spending_deviation_score, 0.75),
    median_spending_deviation_score = median(spending_deviation_score),

    avg_time_since_last_transaction = mean(time_since_last_transaction, na.rm = TRUE),
    median_time_since_last_transaction = median(time_since_last_transaction, na.rm = TRUE)
  ) %>%
  collect()

# Convert to a list for easy access
params <- as.list(scaling_params)


### Apply Scaling & Imputation on Test Set

Now apply transformations using `mutate()` and `coalesce()`.


In [ ]:
%%R
test_scaled <- tbl(con, "financial_trx_testing") %>%
  # Standard Scaling
  mutate(
    ss_velocity_score = (velocity_score - !!params$avg_velocity_score) / !!params$std_velocity_score,
    ss_spending_deviation_score = (spending_deviation_score - !!params$avg_spending_deviation_score) / !!params$std_spending_deviation_score
  ) %>%
  # Min-Max Scaling
  mutate(
    min_max_velocity_score = (velocity_score - !!params$min_velocity_score) /
                             nullif(!!params$max_velocity_score - !!params$min_velocity_score, 0),
    min_max_spending_deviation_score = (spending_deviation_score - !!params$min_spending_deviation_score) /
                                       nullif(!!params$max_spending_deviation_score - !!params$min_spending_deviation_score, 0)
  ) %>%
  # Robust Scaling
  mutate(
    rs_velocity_score = (velocity_score - !!params$q50_velocity_score) /
                        nullif(!!params$q75_velocity_score - !!params$q25_velocity_score, 0),
    rs_spending_deviation_score = (spending_deviation_score - !!params$q50_spending_deviation_score) /
                                  nullif(!!params$q75_spending_deviation_score - !!params$q25_spending_deviation_score, 0)
  ) %>%
  # Handle Missing Values
  mutate(
    time_since_last_transaction_with_0 = coalesce(time_since_last_transaction, 0),
    time_since_last_transaction_with_mean = coalesce(time_since_last_transaction, !!params$avg_time_since_last_transaction),
    time_since_last_transaction_with_median = coalesce(time_since_last_transaction, !!params$median_time_since_last_transaction)
  )

# Show SQL
test_scaled %>% show_query()

# Collect results (or write to file)
test_scaled %>% collect(n = 10)  # Preview first 10 rows


>  Use `!!` (from `rlang`) to splice precomputed values from R into SQL.



### Cleanup (Optional)


In [ ]:
%%R
dbDisconnect(con, shutdown = TRUE)


## Summary and Conclusion

By leveraging DuckDB's SQL capabilities, we were able to efficiently preprocess data directly within the database, minimizing data movement and enhancing performance. This approach not only simplifies the workflow but also ensures that transformations are reproducible and auditable. DuckDB's integration with R through packages like `duckdb` and `duckplyr` allows data scientists to harness the power of SQL while maintaining the flexibility of R for further analysis and modeling.


This tutorial uses `duckplyr` to analyze NYC Yellow Taxi Trip Data (`yellow_tripdata_2023-01.parquet`) with `arrow`, `duckdb`, and `dbplyr`.
It covers:

- **Loading**: Registers Parquet file in DuckDB using `arrow::open_dataset()` and `duckdb_register_arrow()`.
- **Filtering**: Filters trips with `passenger_count > 1`.
- **Aggregation**: Computes average distance, total trips, and groups by payment type and pickup hour.
- **Joins**: Joins with `taxi_zone_lookup.csv` for borough data.
- **Window Functions**: Calculates running revenue for January 2023.
- **Exporting**: Saves filtered trips to CSV using `dbplyr::sql_render()`.
- **Large Data**: Queries remote 2023 Parquet files via `httpfs`.
- **Visualization**: Plots revenue by borough with `ggplot2`.
- *Feature Engineering**:


## References

1. [DuckDB](https://duckdb.org/)

2. [duckbd-r](https://github.com/duckdb/duckdb-r)

3. [duckplyr](https://duckplyr.tidyverse.org/)

4. [Basic Feature Engineering with DuckDB](https://duckdb.org/2025/08/15/ml-data-preprocessing.html?spm=a2ty_o01.29997173.0.0.619ac9217IETtc)
